In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv


In [2]:
import pandas as pd
import re
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import stopwords


nltk.download('wordnet')
nltk.download('omw-1.4')

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
df=pd.read_csv('/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv')

In [4]:
df.head(4)

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative


In [5]:
df.shape

(50000, 2)

In [6]:
df.value_counts('sentiment')

sentiment
negative    25000
positive    25000
Name: count, dtype: int64

# Text Cleaning


In [7]:
print("Missing values:")
print(df.isnull().sum())

df = df.dropna()
print(f"\nAfter dropping: {df.shape}")

Missing values:
review       0
sentiment    0
dtype: int64

After dropping: (50000, 2)


In [8]:
print(" ORIGINAL REVIEW Before Cleaning")
print(df['review'][0][:500])

 ORIGINAL REVIEW Before Cleaning
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ


In [9]:
def clean_text(text):
    """
    Clean text: remove HTML tags, special chars, numbers, convert to lowercase
    """
    # Remove HTML tags (<br />, <p>, etc.)
    text = re.sub(r'<[^>]+>', ' ', text)
    
    # Remove everything except letters and spaces
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    
    
    text = text.lower()
    
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [10]:
df['clean_review'] = df['review'].apply(clean_text)



In [11]:
print("ORIGINAL:")
print(df['review'][0][:200])
print("\n" + "="*60 + "\n")
print("CLEANED:")
print(df['clean_review'][0][:200])

ORIGINAL:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me abo


CLEANED:
one of the other reviewers has mentioned that after watching just oz episode you ll be hooked they are right as this is exactly what happened with me the first thing that struck me about oz was its br


In [12]:
df = df[['clean_review', 'sentiment']]

# Tokenization

***Tokenization means splitting text into smaller pieces called tokens — usually words or sentences. It's the first step computers need to understand text, like breaking a sentence into individual words for analysis.***

In [13]:
df.head(2)

,clean_review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production the filming tech...,positive


In [14]:
text = df['clean_review'][0]
words = word_tokenize(text)

print(f"Total words: {len(words)}")
print(f"\nFirst 30 words: {words[:30]}")

Total words: 313

First 30 words: ['one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', 'oz', 'episode', 'you', 'll', 'be', 'hooked', 'they', 'are', 'right', 'as', 'this', 'is', 'exactly', 'what', 'happened', 'with', 'me', 'the', 'first']


In [15]:
# Tokenize all reviews into word lists
df['words'] = df['clean_review'].apply(word_tokenize)


In [16]:
#  sentence tokenization
text = df['clean_review'][100]  
sentences = sent_tokenize(text)

print(f"Total sentences: {len(sentences)}")
print(f"\nFirst 3 sentences:")
for i, sent in enumerate(sentences[:3]):
    print(f"{i+1}. {sent}")

Total sentences: 1

First 3 sentences:
1. this short film that inspired the soon to be full length feature spatula madness is a hilarious piece that contends against similar cartoons yielding multiple writers the short film stars edward the spatula who after being fired from his job joins in the fight against the evil spoons this premise allows for some funny content near the beginning but is barely present for the remainder of the feature this film s minute running time is absorbed by some odd ball comedy and a small musical number unfortunately not much else lies below it the plot that is set up doesn t really have time to show but it s surely follows it plot better than many high budget hollywood films this film is worth watching at least a few times take it for what it is and don t expect a deep story


In [17]:
# Word count from split() vs word_tokenize()
df['word_count_split'] = df['clean_review'].apply(lambda x: len(x.split()))
df['word_count_tokenize'] = df['words'].apply(len)

print(f"Average using split(): {df['word_count_split'].mean():.2f}")
print(f"Average using word_tokenize(): {df['word_count_tokenize'].mean():.2f}")

Average using split(): 234.18
Average using word_tokenize(): 234.25


In [18]:
df[['clean_review', 'words', 'sentiment']]

,clean_review,words,sentiment
0,one of the other reviewers has mentioned that ...,"[one, of, the, other, reviewers, has, mentione...",positive
1,a wonderful little production the filming tech...,"[a, wonderful, little, production, the, filmin...",positive
2,i thought this was a wonderful way to spend ti...,"[i, thought, this, was, a, wonderful, way, to,...",positive
3,basically there s a family where a little boy ...,"[basically, there, s, a, family, where, a, lit...",negative
4,petter mattei s love in the time of money is a...,"[petter, mattei, s, love, in, the, time, of, m...",positive
...,...,...,...
49995,i thought this movie did a down right good job...,"[i, thought, this, movie, did, a, down, right,...",positive
49996,bad plot bad dialogue bad acting idiotic direc...,"[bad, plot, bad, dialogue, bad, acting, idioti...",negative
49997,i am a catholic taught in parochial elementary...,"[i, am, a, catholic, taught, in, parochial, el...",negative
49998,i m going to have to disagree with the previou...,"[i, m, going, to, have, to, disagree, with, th...",negative


#  stemming_vs_lemmatization

***Both reduce words to their base form. Stemming chops word endings quickly (e.g., "running" → "run"), while lemmatization uses dictionary meaning for accurate results (e.g., "better" → "good").***

In [19]:
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (50000, 5)


,clean_review,sentiment,words,word_count_split,word_count_tokenize
0,one of the other reviewers has mentioned that ...,positive,"[one, of, the, other, reviewers, has, mentione...",313,313
1,a wonderful little production the filming tech...,positive,"[a, wonderful, little, production, the, filmin...",160,160
2,i thought this was a wonderful way to spend ti...,positive,"[i, thought, this, was, a, wonderful, way, to,...",167,167
3,basically there s a family where a little boy ...,negative,"[basically, there, s, a, family, where, a, lit...",133,133
4,petter mattei s love in the time of money is a...,positive,"[petter, mattei, s, love, in, the, time, of, m...",228,228


In [20]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

In [21]:
def stem_words(words_list):
    """Apply stemming to a list of words"""
    return [stemmer.stem(word) for word in words_list]


df['stemmed_words'] = df['words'].apply(stem_words)

print("Sample stemmed words:")
df['stemmed_words'][0][:20]

Sample stemmed words:


['one',
 'of',
 'the',
 'other',
 'review',
 'ha',
 'mention',
 'that',
 'after',
 'watch',
 'just',
 'oz',
 'episod',
 'you',
 'll',
 'be',
 'hook',
 'they',
 'are',
 'right']

In [22]:
def lemmatize_words(words_list):
    """Apply lemmatization to a list of words"""
    return [lemmatizer.lemmatize(word) for word in words_list]


df['lemmatized_words'] = df['words'].apply(lemmatize_words)

print("Sample lemmatized words:")
df['lemmatized_words'][0][:20]

Sample lemmatized words:


['one',
 'of',
 'the',
 'other',
 'reviewer',
 'ha',
 'mentioned',
 'that',
 'after',
 'watching',
 'just',
 'oz',
 'episode',
 'you',
 'll',
 'be',
 'hooked',
 'they',
 'are',
 'right']

**Comparing Original vs Stemmed vs Lemmatized**

In [23]:
idx = 0

print("ORIGINAL words:")
print(df['words'][idx][:15])

print("\nSTEMMED words:")
print(df['stemmed_words'][idx][:15])

print("\nLEMMATIZED words:")
print(df['lemmatized_words'][idx][:15])

ORIGINAL words:
['one', 'of', 'the', 'other', 'reviewers', 'has', 'mentioned', 'that', 'after', 'watching', 'just', 'oz', 'episode', 'you', 'll']

STEMMED words:
['one', 'of', 'the', 'other', 'review', 'ha', 'mention', 'that', 'after', 'watch', 'just', 'oz', 'episod', 'you', 'll']

LEMMATIZED words:
['one', 'of', 'the', 'other', 'reviewer', 'ha', 'mentioned', 'that', 'after', 'watching', 'just', 'oz', 'episode', 'you', 'll']


In [24]:
# Convert word lists back to text
df['stemmed_review'] = df['stemmed_words'].apply(lambda x: ' '.join(x))
df['lemmatized_review'] = df['lemmatized_words'].apply(lambda x: ' '.join(x))

print("STEMMED REVIEW:")
print(df['stemmed_review'][0][:300])

print("\n" + "="*60 + "\n")

print("LEMMATIZED REVIEW:")
print(df['lemmatized_review'][0][:300])

STEMMED REVIEW:
one of the other review ha mention that after watch just oz episod you ll be hook they are right as thi is exactli what happen with me the first thing that struck me about oz wa it brutal and unflinch scene of violenc which set in right from the word go trust me thi is not a show for the faint heart


LEMMATIZED REVIEW:
one of the other reviewer ha mentioned that after watching just oz episode you ll be hooked they are right a this is exactly what happened with me the first thing that struck me about oz wa it brutality and unflinching scene of violence which set in right from the word go trust me this is not a show


In [25]:
df[['clean_review', 'stemmed_review', 'lemmatized_review', 'sentiment']]

,clean_review,stemmed_review,lemmatized_review,sentiment
0,one of the other reviewers has mentioned that ...,one of the other review ha mention that after ...,one of the other reviewer ha mentioned that af...,positive
1,a wonderful little production the filming tech...,a wonder littl product the film techniqu is ve...,a wonderful little production the filming tech...,positive
2,i thought this was a wonderful way to spend ti...,i thought thi wa a wonder way to spend time on...,i thought this wa a wonderful way to spend tim...,positive
3,basically there s a family where a little boy ...,basic there s a famili where a littl boy jake ...,basically there s a family where a little boy ...,negative
4,petter mattei s love in the time of money is a...,petter mattei s love in the time of money is a...,petter mattei s love in the time of money is a...,positive
...,...,...,...,...
49995,i thought this movie did a down right good job...,i thought thi movi did a down right good job i...,i thought this movie did a down right good job...,positive
49996,bad plot bad dialogue bad acting idiotic direc...,bad plot bad dialogu bad act idiot direct the ...,bad plot bad dialogue bad acting idiotic direc...,negative
49997,i am a catholic taught in parochial elementary...,i am a cathol taught in parochi elementari sch...,i am a catholic taught in parochial elementary...,negative
49998,i m going to have to disagree with the previou...,i m go to have to disagre with the previou com...,i m going to have to disagree with the previou...,negative


In [26]:
df.head(3)

,clean_review,sentiment,words,word_count_split,word_count_tokenize,stemmed_words,lemmatized_words,stemmed_review,lemmatized_review
0,one of the other reviewers has mentioned that ...,positive,"[one, of, the, other, reviewers, has, mentione...",313,313,"[one, of, the, other, review, ha, mention, tha...","[one, of, the, other, reviewer, ha, mentioned,...",one of the other review ha mention that after ...,one of the other reviewer ha mentioned that af...
1,a wonderful little production the filming tech...,positive,"[a, wonderful, little, production, the, filmin...",160,160,"[a, wonder, littl, product, the, film, techniq...","[a, wonderful, little, production, the, filmin...",a wonder littl product the film techniqu is ve...,a wonderful little production the filming tech...
2,i thought this was a wonderful way to spend ti...,positive,"[i, thought, this, was, a, wonderful, way, to,...",167,167,"[i, thought, thi, wa, a, wonder, way, to, spen...","[i, thought, this, wa, a, wonderful, way, to, ...",i thought thi wa a wonder way to spend time on...,i thought this wa a wonderful way to spend tim...


# Stopwords

***Stopwords are common words like "the", "is", "a", "in" that appear frequently but add little meaning. Removing them reduces noise and helps NLP models focus on important words.***

In [27]:
stop_words = set(stopwords.words('english'))

print(f"Total English stopwords: {len(stop_words)}")
print(f"\nFirst 10 stopwords: {list(stop_words)[:10]}")

Total English stopwords: 198

First 10 stopwords: ['when', 'ourselves', "aren't", 'them', 'ain', 'out', 'did', "it'd", 'couldn', "they'll"]


In [28]:
def remove_stopwords(words_list):
    """Remove stopwords from a list of words"""
    return [word for word in words_list if word not in stop_words]

sample = df['lemmatized_words'][0][:50]
filtered = remove_stopwords(sample)

print(f"Original: {len(sample)} words")
print(f"After removing stopwords: {len(filtered)} words")
print(f"\nOriginal: {sample[:20]}")
print(f"\nFiltered: {filtered[:20]}")

Original: 50 words
After removing stopwords: 22 words

Original: ['one', 'of', 'the', 'other', 'reviewer', 'ha', 'mentioned', 'that', 'after', 'watching', 'just', 'oz', 'episode', 'you', 'll', 'be', 'hooked', 'they', 'are', 'right']

Filtered: ['one', 'reviewer', 'ha', 'mentioned', 'watching', 'oz', 'episode', 'hooked', 'right', 'exactly', 'happened', 'first', 'thing', 'struck', 'oz', 'wa', 'brutality', 'unflinching', 'scene', 'violence']


In [29]:
# Apply to all reviews
df['filtered_words'] = df['lemmatized_words'].apply(remove_stopwords)

print("Sample after removing stopwords:")
df['filtered_words'][0][:20]

Sample after removing stopwords:


['one',
 'reviewer',
 'ha',
 'mentioned',
 'watching',
 'oz',
 'episode',
 'hooked',
 'right',
 'exactly',
 'happened',
 'first',
 'thing',
 'struck',
 'oz',
 'wa',
 'brutality',
 'unflinching',
 'scene',
 'violence']

In [30]:
# Convert filtered words back to text
df['filtered_review'] = df['filtered_words'].apply(lambda x: ' '.join(x))

print("FILTERED REVIEW (no stopwords):")
print(df['filtered_review'][0][:400])

FILTERED REVIEW (no stopwords):
one reviewer ha mentioned watching oz episode hooked right exactly happened first thing struck oz wa brutality unflinching scene violence set right word go trust show faint hearted timid show pull punch regard drug sex violence hardcore classic use word called oz nickname given oswald maximum security state penitentary focus mainly emerald city experimental section prison cell glass front face inw


In [31]:
# Count words before and after
df['words_before'] = df['lemmatized_words'].apply(len)
df['words_after'] = df['filtered_words'].apply(len)

print(f"Average words before: {df['words_before'].mean():.2f}")
print(f"Average words after: {df['words_after'].mean():.2f}")
print(f"\nWords removed per review: {df['words_before'].mean() - df['words_after'].mean():.2f}")
print(f"Percentage removed: {(df['words_before'].mean() - df['words_after'].mean()) / df['words_before'].mean() * 100:.1f}%")

Average words before: 234.25
Average words after: 120.89

Words removed per review: 113.36
Percentage removed: 48.4%


In [32]:
df[['clean_review', 'stemmed_review', 'lemmatized_review','filtered_review', 'filtered_words', 'sentiment']].head(3)

,clean_review,stemmed_review,lemmatized_review,filtered_review,filtered_words,sentiment
0,one of the other reviewers has mentioned that ...,one of the other review ha mention that after ...,one of the other reviewer ha mentioned that af...,one reviewer ha mentioned watching oz episode ...,"[one, reviewer, ha, mentioned, watching, oz, e...",positive
1,a wonderful little production the filming tech...,a wonder littl product the film techniqu is ve...,a wonderful little production the filming tech...,wonderful little production filming technique ...,"[wonderful, little, production, filming, techn...",positive
2,i thought this was a wonderful way to spend ti...,i thought thi wa a wonder way to spend time on...,i thought this wa a wonderful way to spend tim...,thought wa wonderful way spend time hot summer...,"[thought, wa, wonderful, way, spend, time, hot...",positive
